# Stack Compass — Análisis narrado

**Pregunta:** ¿Qué stacks tecnológicos maximizan el trinomio salario / satisfacción / empleabilidad para un perfil data-IA junior en la UE?

**Fuente:** Stack Overflow Annual Developer Survey (año más reciente disponible).  
**Población:** Roles data/IA (núcleo + frontera) en los 27 países de la UE.

In [1]:
import sys
sys.path.insert(0, "../src")

from load import load_raw, get_duckdb_connection, load_processed
from clean import filter_population, clean_salary, normalize_multi_valued, filter_geography, add_region
from metrics import salary_by_stack, satisfaction_by_stack, employability_by_stack

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Carga de datos

Leemos el CSV crudo sin inferencia de tipos (`low_memory=False`) para evitar advertencias en columnas mixtas.

In [2]:
df_raw = load_raw()
print(f"Filas: {len(df_raw):,}  |  Columnas: {df_raw.shape[1]}")
df_raw.head(2)

Filas: 49,191  |  Columnas: 172


,ResponseId,MainBranch,Age,EdLevel,Employment,EmploymentAddl,WorkExp,LearnCodeChoose,LearnCode,LearnCodeAI,...,AIAgentOrchestration,AIAgentOrchWrite,AIAgentObserveSecure,AIAgentObsWrite,AIAgentExternal,AIAgentExtWrite,AIHuman,AIOpen,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,"Caring for dependents (children, elderly, etc.)",8.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,Vertex AI,NaN,NaN,NaN,ChatGPT,NaN,When I don’t trust AI’s answers,"Troubleshooting, profiling, debugging",61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,NaN,2.0,"Yes, I am not new to coding but am learning ne...",Online Courses or Certification (includes all ...,"Yes, I learned how to use AI-enabled tools for...",...,NaN,NaN,NaN,NaN,NaN,NaN,When I don’t trust AI’s answers;When I want to...,All skills. AI is a flop.,104413.0,9.0


## 2. Limpieza y filtrado de población

Nos quedamos con respondentes en la UE que declaran roles data/IA.  
Después limpiamos la columna salarial eliminando nulos y outliers extremos (p1–p99).

In [5]:
pob = filter_population(df_raw)
pob = filter_geography(pob)
pob = add_region(pob)
print(f"Población filtrada: {len(pob):,} filas")

pob_sal = clean_salary(pob)
print(f"NaN en salario: {pob_sal['ConvertedCompYearly'].isna().sum()}")
print(f"Con salario válido (p1-p99): {len(pob_sal):,} filas")

Población filtrada: 1,194 filas
NaN en salario: 297
Con salario válido (p1-p99): 1,194 filas


## 3. Métricas del trinomio

Calculamos las tres dimensiones de interés sobre la misma población.

### 3.1 Salario por stack

Mediana de `ConvertedCompYearly` (USD/año) agrupada por tecnología principal.

In [4]:
df_sal = salary_by_stack(pob_sal)
df_sal.head(10)

AttributeError: 'NoneType' object has no attribute 'head'

### 3.2 Satisfacción por stack

Puntuación media de `JobSat` por tecnología. Escala ordinal de la encuesta.

In [ ]:
df_sat = satisfaction_by_stack(pob)
df_sat.head(10)

### 3.3 Empleabilidad por stack

Frecuencia de aparición en `LanguageHaveWorkedWith` dentro de la población filtrada.  
Mayor frecuencia → más posiciones que demandan ese stack.

In [ ]:
pob_exp = normalize_multi_valued(pob, col="LanguageHaveWorkedWith")
df_emp = employability_by_stack(pob_exp)
df_emp.head(10)

## 4. Conclusiones

> *Sección pendiente de rellenar una vez las métricas estén implementadas.*